In [22]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import RandomizedSearchCV
from datetime import datetime
current_year = datetime.now().year

In [23]:
df = pd.read_csv("data.csv")

In [24]:
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


In [25]:
df.shape

(4600, 18)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   object 
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   object 
 15  city           4600 non-null   object 
 16  statezip       4600 non-null   object 
 17  country        4600 non-null   object 
dtypes: float

In [27]:
df.columns

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='object')

In [28]:
df.isnull().sum()

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

In [29]:
print((df['price'] == 0).sum())

49


In [30]:
df = df[df['price'] != 0]

In [31]:
print((df['price'] == 0).sum())

0


In [32]:
df['house_age'] = current_year - df['yr_built']

In [33]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

df = df[(df["price"]>=lower) & (df["price"]<=upper)]

In [34]:
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country,house_age
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA,71
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA,60
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA,63
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA,50
5,2014-05-02 00:00:00,490000.0,2.0,1.00,880,6380,1.0,0,0,3,880,0,1938,1994,522 NE 88th St,Seattle,WA 98115,USA,88


In [35]:
df.columns

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country', 'house_age'],
      dtype='object')

# Data Preprocessing

In [36]:
df = pd.get_dummies(
            df,
            columns = ['city','statezip'],
            drop_first = True
)

In [37]:
df.shape

(4311, 136)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4311 entries, 0 to 4599
Columns: 136 entries, date to statezip_WA 98354
dtypes: bool(119), float64(4), int64(10), object(3)
memory usage: 1.1+ MB


# Split the dataset

In [39]:
X = df.drop(['price','date','country','street'],axis = 1)
y = df['price']

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

# Linear Regression 

In [41]:
model_LR = LinearRegression()

In [42]:
model_LR = model_LR.fit(X_train, y_train)

In [43]:
y_pred_lr = model_LR.predict(X_test)

In [44]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

In [45]:
print(f"Mean Absolute Error: ",mae_lr)
print(f"Mean Squared Error: ",mse_lr)
print(f"Root Mean Squared Error: ",rmse_lr)
print(f"R2-score: ",r2_lr)

Mean Absolute Error:  71766.88304908549
Mean Squared Error:  10315794421.953981
Root Mean Squared Error:  101566.69937511005
R2-score:  0.7799430170150236


# Support Vector Regression

In [46]:
model_svm = SVR(kernel = 'rbf')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [47]:
model_svm.fit(X_train_scaled, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [48]:
y_pred_svr = model_svm.predict(X_test_scaled)

In [49]:
mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
r2_svr  = r2_score(y_test, y_pred_svr)

In [50]:
print(f"Mean Absolute Error: ",mae_svr)
print(f"Mean Squared Error: ",mse_svr)
print(f"Root Mean Squared Error: ",rmse_svr)
print(f"R2-score: ",r2_svr)

Mean Absolute Error:  170546.2831204251
Mean Squared Error:  48757308089.98099
Root Mean Squared Error:  220810.5706029061
R2-score:  -0.04009305322293155


# Decision Tree

In [51]:
model_dt = DecisionTreeRegressor(
            max_depth=10,
            min_samples_leaf=5,
            random_state=42
)

In [52]:
model_dt.fit(X_train,y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [53]:
y_pred_dt = model_dt.predict(X_test)

In [54]:
mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

In [55]:
print(f"Mean Absolute Error:",mae_dt)
print(f"Mean Squared Error:",mse_dt)
print(f"Root Mean Squared Error:",rmse_dt)
print(f"R2-score:",r2_dt)

Mean Absolute Error: 100343.55501278717
Mean Squared Error: 18784825720.899
Root Mean Squared Error: 137057.74593542315
R2-score: 0.5992812666717894


# Random Forest

In [56]:
params = {
    "n_estimators":[200,300,500,800],
    "max_depth":[10,20,30,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
}

In [57]:
model_rf = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=params,
    n_iter=100,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs = -1,
    verbose = 2
)

In [58]:
model_rf.fit(X_train, y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'max_depth': [10, 20, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [200, 300, ...]}"
,n_iter,100
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [59]:
print(model_rf.best_params_)

{'n_estimators': 800, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None}


In [60]:
print(model_rf.best_score_)

0.7261849789388902


In [61]:
best_rf = model_rf.best_estimator_

In [62]:
y_pred_rf = best_rf.predict(X_test)

In [63]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

In [64]:
print(f"Mean Absolute Error:",mae_rf)
print(f"Mean Squared Error:",mse_rf)
print(f"Root Mean Squared Error:",rmse_rf)
print(f"R2-score:",r2_rf)

Mean Absolute Error: 76837.5274505198
Mean Squared Error: 12599146111.598223
Root Mean Squared Error: 112245.9180175307
R2-score: 0.731234457754923


In [65]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'SVR', 'Decision Tree', 'Random Forest'],
    'MAE': [mae_lr, mae_svr, mae_dt, mae_rf],
    'MSE': [mse_lr, mse_svr, mse_dt, mse_rf],
    'RMSE': [rmse_lr, rmse_svr, rmse_dt, rmse_rf],
    'R2': [r2_lr, r2_svr, r2_dt, r2_rf]
})
results.sort_values('R2', ascending=False)

,Model,MAE,MSE,RMSE,R2
0,Linear Regression,71766.883049,1.031579e+10,101566.699375,0.779943
3,Random Forest,76837.527451,1.259915e+10,112245.918018,0.731234
2,Decision Tree,100343.555013,1.878483e+10,137057.745935,0.599281
1,SVR,170546.283120,4.875731e+10,220810.570603,-0.040093
